Key formulas used in implementation:

- Steering vector (half-wavelength ULA):
$
\mathbf{a}(\theta)=\left[e^{-j2\pi d m\sin\theta}\right]_{m=0}^{M-1},\ d=0.5
$.

- Angle grid and true manifold:
$
\Theta_g=\{\theta_g^{(k)}\},\ \mathbf{A}_0=[\mathbf{a}(\theta_{0,1}),\mathbf{a}(\theta_{0,2})]
$.

- Sample covariance:
$
\hat{\mathbf{R}}_x=\frac{1}{L}\mathbf{X}\mathbf{X}^H
$.

- MUSIC pseudo-spectrum:
$
P_{\mathrm{MU}}(\theta)=\frac{1}{\mathbf{a}^H(\theta)\mathbf{U}_n\mathbf{U}_n^H\mathbf{a}(\theta)}
$,
with $\mathbf{U}_n$ from the smallest $M-D$ eigenvectors of $\hat{\mathbf{R}}_x$.

- Root-MUSIC polynomial:
$
Q(z)=\sum_{p=-(M-1)}^{M-1}c_p z^{-p},\ c_p=\sum_{m-n=p}[\mathbf{U}_n\mathbf{U}_n^H]_{m,n}
$.

- ESPRIT shift invariance:
$
\mathbf{\Psi}=(\mathbf{J}_1\mathbf{U}_s)^\dagger(\mathbf{J}_2\mathbf{U}_s),\ 
\hat{\theta}_i=\arcsin\!\left(-\frac{\arg(\lambda_i(\mathbf{\Psi}))}{2\pi d}\right)
$.

- DML objective:
$
\hat{\boldsymbol{\theta}}_{\mathrm{DML}}=\arg\min_{\boldsymbol{\theta}}
\operatorname{tr}\!\left(\mathbf{P}_{\mathbf{A}(\boldsymbol{\theta})}^{\perp}\hat{\mathbf{R}}_x\right)
$.

- SML objective:
$
\hat{\boldsymbol{\theta}}_{\mathrm{SML}}=\arg\min_{\boldsymbol{\theta},\mathbf{R}_s,\sigma_n^2}
\left[\log\det\mathbf{R}_x+\operatorname{tr}(\mathbf{R}_x^{-1}\hat{\mathbf{R}}_x)\right]
$,
with iterative local refinement and convergence check.

- Stochastic CRB:
$
[\mathbf{J}]_{ij}=2L\Re\!\left\{\operatorname{tr}\!\left(
\mathbf{R}_x^{-1}\frac{\partial \mathbf{R}_x}{\partial\theta_i}
\mathbf{R}_x^{-1}\frac{\partial \mathbf{R}_x}{\partial\theta_j}\right)\right\},\ 
\mathrm{CRB}=\mathbf{J}^{-1}
$.

- Metrics:
$
\mathrm{RMSE}=\sqrt{\frac{1}{DT}\sum_{t=1}^T\|\hat{\boldsymbol{\theta}}_t-\boldsymbol{\theta}_0\|_2^2}\cdot\frac{180}{\pi}
$,
$
P_{\mathrm{res}}=\frac{1}{T}\sum_{t=1}^T\mathbf{1}\{\mathcal{R}_t\}
$,
threshold rule $
\mathrm{RMSE}>3\times \mathrm{CRB}_{\mathrm{RMSE}}
$.

In [ ]:
import time
import json
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import find_peaks
from scipy.linalg import block_diag

Next block defines simulation data-generation utilities for the array model.

**Computes:**
- ULA steering vectors/matrices,
- angle grid construction,
- snapshot synthesis $ \mathbf{X}=\mathbf{A}_0\mathbf{S}+\mathbf{N} $ with SNR-consistent complex AWGN,
- stochastic CRB per SNR.

**Key inputs/outputs:**
- Inputs: $M, D, L, \boldsymbol{\theta}_0, \mathrm{SNR}_{\mathrm{dB}}$
- Outputs: $\Theta_g, \mathbf{A}_0, \mathbf{X}, \hat{\mathbf{R}}_x, \sigma_n^2, \mathrm{CRB}$

**Equations:**
$
\hat{\mathbf{R}}_x=\frac{1}{L}\mathbf{X}\mathbf{X}^H,\ 
\mathbf{n}=\frac{\sigma}{\sqrt{2}}(\mathbf{n}_r+j\mathbf{n}_i),\
\mathbf{R}_x=\mathbf{A}_0\mathbf{R}_s\mathbf{A}_0^H+\sigma_n^2\mathbf{I}
$.

In [ ]:
def steering_vector(theta_rad, num_rx, d_over_lambda=0.5):
    m = np.arange(num_rx)
    return np.exp(-1j * 2 * np.pi * d_over_lambda * m * np.sin(theta_rad))


def steering_matrix(theta_rad_vec, num_rx, d_over_lambda=0.5):
    A = np.column_stack([steering_vector(th, num_rx, d_over_lambda) for th in theta_rad_vec])
    return A


def build_angle_grid(theta_min_deg, theta_max_deg, grid_step_deg):
    theta_grid_deg = np.arange(theta_min_deg, theta_max_deg + 1e-12, grid_step_deg)
    theta_grid_rad = np.deg2rad(theta_grid_deg)
    return theta_grid_deg, theta_grid_rad


def generate_snapshots(
    snr_db,
    num_snapshots,
    theta_true_rad,
    num_rx,
    num_sources,
    d_over_lambda=0.5,
    sigma_s2=1.0,
):
    A0 = steering_matrix(theta_true_rad, num_rx, d_over_lambda)
    assert A0.shape == (num_rx, num_sources), "steering matrix shape mismatch"

    # Source snapshots: S_{d,l} ~ CN(0, sigma_s2)
    S = np.sqrt(sigma_s2 / 2.0) * (
        np.random.randn(num_sources, num_snapshots) + 1j * np.random.randn(num_sources, num_snapshots)
    )

    X_signal = A0 @ S

    # Step-2 compliant per-sensor SNR: sigma_n^2 = D*sigma_s^2 / gamma
    gamma = 10 ** (snr_db / 10.0)
    sigma_n2 = (num_sources * sigma_s2) / gamma
    sigma_n = np.sqrt(sigma_n2)

    # Complex AWGN with total variance sigma_n2
    N = (sigma_n / np.sqrt(2.0)) * (
        np.random.randn(num_rx, num_snapshots) + 1j * np.random.randn(num_rx, num_snapshots)
    )

    X = X_signal + N
    Rhat = (X @ X.conj().T) / num_snapshots
    assert Rhat.shape == (num_rx, num_rx), "covariance shape mismatch"

    if num_snapshots < num_rx:
        loading = 1e-3 * np.real(np.trace(Rhat)) / num_rx
        Rhat = Rhat + loading * np.eye(num_rx)

    return X, Rhat, sigma_n2, A0


def stochastic_crb_two_source(
    snr_db,
    theta_true_rad,
    num_rx,
    num_snapshots,
    d_over_lambda=0.5,
    sigma_s2=1.0,
):
    D = len(theta_true_rad)
    A0 = steering_matrix(theta_true_rad, num_rx, d_over_lambda)
    Rs = sigma_s2 * np.eye(D)

    gamma = 10 ** (snr_db / 10.0)
    sigma_n2 = (D * sigma_s2) / gamma

    Rx = A0 @ Rs @ A0.conj().T + sigma_n2 * np.eye(num_rx)
    Rx_inv = np.linalg.inv(Rx)

    m = np.arange(num_rx)
    dA = []
    for th in theta_true_rad:
        a = np.exp(-1j * 2 * np.pi * d_over_lambda * m * np.sin(th))
        da = a * (-1j * 2 * np.pi * d_over_lambda * m * np.cos(th))
        dA.append(da)

    dR = []
    for i in range(D):
        a_i = A0[:, i : i + 1]
        da_i = dA[i][:, None]
        dR_i = sigma_s2 * (da_i @ a_i.conj().T + a_i @ da_i.conj().T)
        dR.append(dR_i)

    J = np.zeros((D, D), dtype=float)
    for i in range(D):
        for j in range(D):
            J[i, j] = 2.0 * num_snapshots * np.real(
                np.trace(Rx_inv @ dR[i] @ Rx_inv @ dR[j])
            )

    J = J + 1e-12 * np.eye(D)
    CRB = np.linalg.inv(J)
    crb_rmse_rad = np.sqrt(np.trace(CRB) / D)
    crb_rmse_deg = np.rad2deg(crb_rmse_rad)
    return CRB, crb_rmse_deg, sigma_n2

Next block implements all estimators (proposed + baselines) and supporting objective functions.

**Computes:**
- MUSIC pseudo-spectrum and peak localization,
- Root-MUSIC root selection,
- ESPRIT shift-invariance estimate,
- DML grid-search minimization,
- SML iterative local likelihood minimization with convergence history.

**Key inputs/outputs:**
- Inputs: $\hat{\mathbf{R}}_x$, $D$, $\Theta_g$, $M$
- Outputs: $\hat{\boldsymbol{\theta}}_{\text{music}}, \hat{\boldsymbol{\theta}}_{\text{root}}, \hat{\boldsymbol{\theta}}_{\text{esprit}}, \hat{\boldsymbol{\theta}}_{\text{dml}}, \hat{\boldsymbol{\theta}}_{\text{sml}}$

**Equations:**
$
P_{\mathrm{MU}}(\theta)=1/\Re\{\mathbf{a}^H\mathbf{U}_n\mathbf{U}_n^H\mathbf{a}\},\ 
\hat{\boldsymbol{\theta}}_{\mathrm{DML}}=\arg\min \operatorname{tr}(\mathbf{P}_A^\perp\hat{\mathbf{R}}_x),\
\hat{\boldsymbol{\theta}}_{\mathrm{SML}}=\arg\min \log\det\mathbf{R}_x+\operatorname{tr}(\mathbf{R}_x^{-1}\hat{\mathbf{R}}_x)
$.

In [ ]:
def _sort_theta(theta_rad):
    return np.sort(np.real(theta_rad))


def _pair_error_rad(theta_hat_rad, theta_true_rad):
    return _sort_theta(theta_hat_rad) - _sort_theta(theta_true_rad)


def _resolution_success(theta_hat_rad, theta_true_rad, tol_deg=2.0, require_dominant_two=True, dominant_two_exists=True):
    if theta_hat_rad is None or len(theta_hat_rad) != len(theta_true_rad):
        return 0
    if require_dominant_two and (not dominant_two_exists):
        return 0
    tol = np.deg2rad(tol_deg)
    th_hat = _sort_theta(theta_hat_rad)
    th_true = _sort_theta(theta_true_rad)
    nearest = np.min(np.abs(th_hat[:, None] - th_true[None, :]), axis=1)
    return int(np.all(nearest <= tol))


def music_spectrum_and_estimate(Rhat, num_sources, theta_grid_rad, num_rx, d_over_lambda=0.5):
    eigvals, eigvecs = np.linalg.eigh(Rhat)  # ascending eigenvalues
    E_n = eigvecs[:, :-num_sources]
    assert E_n.shape == (num_rx, num_rx - num_sources), "subspace shape wrong"

    A_theta = steering_matrix(theta_grid_rad, num_rx, d_over_lambda)
    Pn = E_n @ E_n.conj().T
    denom = np.sum(np.conj(A_theta) * (Pn @ A_theta), axis=0).real
    denom = np.maximum(denom, 1e-12)
    P_music = 1.0 / denom

    peaks, _ = find_peaks(P_music)
    dominant_two_exists = len(peaks) >= 2
    if len(peaks) >= 2:
        strongest = peaks[np.argsort(P_music[peaks])[-num_sources:]]
    else:
        strongest = np.argsort(P_music)[-num_sources:]
    strongest = np.sort(strongest)
    theta_hat = _sort_theta(theta_grid_rad[strongest])

    return theta_hat, P_music, strongest, dominant_two_exists, E_n, eigvals


def root_music_estimate(Rhat, num_sources, num_rx, d_over_lambda=0.5):
    eigvals, eigvecs = np.linalg.eigh(Rhat)
    E_n = eigvecs[:, :-num_sources]
    assert E_n.shape == (num_rx, num_rx - num_sources), "subspace shape wrong"
    Pn = E_n @ E_n.conj().T

    c_p = []
    for p in range(-(num_rx - 1), num_rx):
        c_p.append(np.sum(np.diag(Pn, k=p)))
    coeff = np.array(c_p, dtype=np.complex128)

    roots = np.roots(coeff)
    roots_in = roots[np.abs(roots) < 1.0]
    candidates = roots_in if len(roots_in) >= num_sources else roots
    idx = np.argsort(np.abs(np.abs(candidates) - 1.0))[:num_sources]
    z_hat = candidates[idx]

    sin_theta = -np.angle(z_hat) / (2.0 * np.pi * d_over_lambda)
    sin_theta = np.clip(np.real(sin_theta), -1.0, 1.0)
    theta_hat = _sort_theta(np.arcsin(sin_theta))
    return theta_hat


def esprit_estimate(Rhat, num_sources, num_rx, d_over_lambda=0.5):
    eigvals, eigvecs = np.linalg.eigh(Rhat)
    U_s = eigvecs[:, -num_sources:]

    J1 = np.hstack([np.eye(num_rx - 1), np.zeros((num_rx - 1, 1))])
    J2 = np.hstack([np.zeros((num_rx - 1, 1)), np.eye(num_rx - 1)])

    Psi = np.linalg.pinv(J1 @ U_s) @ (J2 @ U_s)
    lam = np.linalg.eigvals(Psi)

    sin_theta = -np.angle(lam) / (2.0 * np.pi * d_over_lambda)
    sin_theta = np.clip(np.real(sin_theta), -1.0, 1.0)
    theta_hat = _sort_theta(np.arcsin(sin_theta))
    return theta_hat


def dml_cost(theta_pair_rad, Rhat, num_rx, d_over_lambda=0.5):
    A = steering_matrix(theta_pair_rad, num_rx, d_over_lambda)
    AhA = A.conj().T @ A
    if np.linalg.cond(AhA) > 1e10:
        return np.inf
    P_A = A @ np.linalg.inv(AhA) @ A.conj().T
    P_perp = np.eye(num_rx) - P_A
    return float(np.real(np.trace(P_perp @ Rhat)))


def dml_estimate(Rhat, num_sources, num_rx, theta_grid_rad, music_spectrum, d_over_lambda=0.5):
    peaks, _ = find_peaks(music_spectrum)
    if len(peaks) < 4:
        peaks = np.argsort(music_spectrum)[-12:]
    peak_order = peaks[np.argsort(music_spectrum[peaks])[::-1]]
    cand_idx = peak_order[: min(10, len(peak_order))]
    cand_angles = np.unique(theta_grid_rad[cand_idx])

    if len(cand_angles) < 2:
        cand_angles = theta_grid_rad[np.argsort(music_spectrum)[-20:]]

    best_cost = np.inf
    best_pair = None
    for i in range(len(cand_angles)):
        for j in range(i + 1, len(cand_angles)):
            pair = np.array([cand_angles[i], cand_angles[j]])
            c = dml_cost(pair, Rhat, num_rx, d_over_lambda)
            if c < best_cost:
                best_cost = c
                best_pair = pair

    if best_pair is None:
        best_pair = _sort_theta(theta_grid_rad[np.argsort(music_spectrum)[-2:]])

    fine_grid = np.deg2rad(np.arange(-1.5, 1.51, 0.3))
    local_candidates_1 = best_pair[0] + fine_grid
    local_candidates_2 = best_pair[1] + fine_grid

    for a1 in local_candidates_1:
        for a2 in local_candidates_2:
            if a2 <= a1 + np.deg2rad(0.5):
                continue
            pair = np.array([a1, a2])
            c = dml_cost(pair, Rhat, num_rx, d_over_lambda)
            if c < best_cost:
                best_cost = c
                best_pair = pair

    return _sort_theta(best_pair)


def sml_nll_with_nuisance(theta_pair_rad, Rhat, num_rx, num_sources, d_over_lambda=0.5):
    A = steering_matrix(theta_pair_rad, num_rx, d_over_lambda)
    A_pinv = np.linalg.pinv(A)

    Rs_hat = A_pinv @ Rhat @ A_pinv.conj().T
    Rs_hat = (Rs_hat + Rs_hat.conj().T) / 2.0
    w, V = np.linalg.eigh(Rs_hat)
    w = np.clip(np.real(w), 1e-8, None)
    Rs_psd = V @ np.diag(w) @ V.conj().T

    P_A = A @ np.linalg.pinv(A.conj().T @ A) @ A.conj().T
    P_perp = np.eye(num_rx) - P_A
    sigma_n2 = np.real(np.trace(P_perp @ Rhat)) / max(num_rx - num_sources, 1)
    sigma_n2 = float(max(sigma_n2, 1e-8))

    Rx_model = A @ Rs_psd @ A.conj().T + sigma_n2 * np.eye(num_rx)
    Rx_model = (Rx_model + Rx_model.conj().T) / 2.0 + 1e-9 * np.eye(num_rx)

    sign, logdet = np.linalg.slogdet(Rx_model)
    if sign <= 0:
        return np.inf, sigma_n2, Rs_psd

    Rx_inv = np.linalg.inv(Rx_model)
    nll = float(np.real(logdet + np.trace(Rx_inv @ Rhat)))
    return nll, sigma_n2, Rs_psd


def sml_estimate_iterative(
    Rhat,
    init_theta_rad,
    num_sources,
    num_rx,
    d_over_lambda=0.5,
    max_iter=15,
    tol=1e-4,
):
    theta_old = _sort_theta(init_theta_rad.copy())
    obj_hist = []
    delta = np.deg2rad(2.0)

    for _ in range(max_iter):
        theta_new = theta_old.copy()

        for i in range(num_sources):
            scan = np.linspace(theta_old[i] - delta, theta_old[i] + delta, 11)
            best_cost = np.inf
            best_val = theta_old[i]

            for val in scan:
                candidate = theta_new.copy()
                candidate[i] = val
                candidate = _sort_theta(candidate)
                if np.min(np.diff(candidate)) < np.deg2rad(0.5):
                    continue
                c, _, _ = sml_nll_with_nuisance(candidate, Rhat, num_rx, num_sources, d_over_lambda)
                if c < best_cost:
                    best_cost = c
                    best_val = val

            theta_new[i] = best_val
            theta_new = _sort_theta(theta_new)

        obj_val, _, _ = sml_nll_with_nuisance(theta_new, Rhat, num_rx, num_sources, d_over_lambda)
        obj_hist.append(obj_val)

        rel_change = np.linalg.norm(theta_new - theta_old) / (np.linalg.norm(theta_old) + 1e-12)
        theta_old = theta_new
        delta *= 0.7
        if rel_change < tol:
            break

    return _sort_theta(theta_old), obj_hist

Next block assembles the full Monte Carlo evaluation sweep over SNR and computes all performance curves.

**Computes:**
- `run_evaluation(eval_config)`:
  - loops through every SNR point,
  - regenerates data at each operating point/trial,
  - runs MUSIC + all baselines independently,
  - computes RMSE, resolution probability, runtime, CRB ratio, threshold masks,
  - computes snapshot sensitivity curves.

**Key inputs/outputs:**
- Input: `eval_config` with `snr_points`, `num_trials`, `snapshot_points`, etc.
- Output: JSON-serializable `performance_data` plus diagnostics.

**Equations:**
$
\mathrm{RMSE}_q=\sqrt{\frac{1}{DT}\sum_t\|\mathbf{e}_{q,t}\|_2^2}\frac{180}{\pi},\ 
\hat{P}_{\mathrm{res},q}=\frac{1}{T}\sum_t\mathcal{R}_{q,t},\
\mathrm{SNR}_{\mathrm{th}}=\min\{\mathrm{SNR}_q:\hat{P}_{\mathrm{res},q}\ge0.9\}
$.

In [ ]:
def run_evaluation(eval_config):
    SNR_POINTS_DB = [float(v) for v in eval_config.get("snr_points", eval_config.get("variable_points", []))]
    NUM_TRIALS = int(eval_config.get("num_trials", 1000))
    NUM_RX = int(eval_config.get("num_rx", 8))
    NUM_SOURCES = int(eval_config.get("num_sources", 2))
    NUM_SNAPSHOTS = int(eval_config.get("num_snapshots", 200))
    D_OVER_LAMBDA = float(eval_config.get("d_over_lambda", 0.5))
    THETA_TRUE_DEG = np.array(eval_config.get("theta_true_deg", [-7.5, 7.5]), dtype=float)
    THETA_TRUE_RAD = np.deg2rad(THETA_TRUE_DEG)
    GRID_MIN_DEG, GRID_MAX_DEG = eval_config.get("scan_range_deg", [-90, 90])
    GRID_STEP_DEG = float(eval_config.get("grid_step_deg", 0.1))
    RES_TOL_DEG = float(eval_config.get("resolution_tol_deg", 2.0))
    RES_TARGET = float(eval_config.get("resolution_prob_target", 0.9))
    DIAG_SNR_DB = float(eval_config.get("diagnostic_snr_db", 20.0))
    SNAPSHOT_POINTS = [int(v) for v in eval_config.get("snapshot_points", [50, 100, 200, 400])]
    NUM_TRIALS_SENS = int(eval_config.get("num_trials_sensitivity", max(200, NUM_TRIALS // 5)))

    theta_grid_deg, theta_grid_rad = build_angle_grid(GRID_MIN_DEG, GRID_MAX_DEG, GRID_STEP_DEG)

    method_names = ["music", "root_music", "esprit", "dml", "sml"]
    rmse_curve = {m: [] for m in method_names}
    pres_curve = {m: [] for m in method_names}
    runtime_curve_ms = {m: [] for m in method_names}
    ratio_curve = {m: [] for m in method_names}
    threshold_mask = {m: [] for m in method_names}
    crb_curve_deg = []

    diag_spectrum = None
    diag_spectrum_db = None
    diag_obj_hist_sml = []

    for snr_db in SNR_POINTS_DB:
        err_trials = {m: [] for m in method_names}
        res_trials = {m: [] for m in method_names}
        runtime_acc = {m: 0.0 for m in method_names}

        for t in range(NUM_TRIALS):
            X, Rhat, sigma_n2_emp, A0 = generate_snapshots(
                snr_db=snr_db,
                num_snapshots=NUM_SNAPSHOTS,
                theta_true_rad=THETA_TRUE_RAD,
                num_rx=NUM_RX,
                num_sources=NUM_SOURCES,
                d_over_lambda=D_OVER_LAMBDA,
                sigma_s2=1.0,
            )

            # Fixed root-cause bug: this function returns 6 values (not 4)
            t0 = time.perf_counter()
            theta_music, P_music, peak_idx, dom_two, E_n, eigvals = music_spectrum_and_estimate(
                Rhat, NUM_SOURCES, theta_grid_rad, NUM_RX, D_OVER_LAMBDA
            )
            runtime_acc["music"] += time.perf_counter() - t0

            t0 = time.perf_counter()
            theta_root = root_music_estimate(Rhat, NUM_SOURCES, NUM_RX, D_OVER_LAMBDA)
            runtime_acc["root_music"] += time.perf_counter() - t0

            t0 = time.perf_counter()
            theta_esprit = esprit_estimate(Rhat, NUM_SOURCES, NUM_RX, D_OVER_LAMBDA)
            runtime_acc["esprit"] += time.perf_counter() - t0

            t0 = time.perf_counter()
            theta_dml = dml_estimate(Rhat, NUM_SOURCES, NUM_RX, theta_grid_rad, P_music, D_OVER_LAMBDA)
            runtime_acc["dml"] += time.perf_counter() - t0

            t0 = time.perf_counter()
            theta_sml, obj_hist = sml_estimate_iterative(
                Rhat=Rhat,
                init_theta_rad=theta_dml,
                num_sources=NUM_SOURCES,
                num_rx=NUM_RX,
                d_over_lambda=D_OVER_LAMBDA,
                max_iter=15,
                tol=1e-4,
            )
            runtime_acc["sml"] += time.perf_counter() - t0

            if (diag_spectrum is None) and (abs(snr_db - DIAG_SNR_DB) < 1e-12):
                diag_spectrum = P_music.copy()
                diag_spectrum_db = 10.0 * np.log10(P_music / np.max(P_music))
                diag_obj_hist_sml = obj_hist.copy()

            err_trials["music"].append(_pair_error_rad(theta_music, THETA_TRUE_RAD))
            err_trials["root_music"].append(_pair_error_rad(theta_root, THETA_TRUE_RAD))
            err_trials["esprit"].append(_pair_error_rad(theta_esprit, THETA_TRUE_RAD))
            err_trials["dml"].append(_pair_error_rad(theta_dml, THETA_TRUE_RAD))
            err_trials["sml"].append(_pair_error_rad(theta_sml, THETA_TRUE_RAD))

            res_trials["music"].append(_resolution_success(theta_music, THETA_TRUE_RAD, RES_TOL_DEG, True, dom_two))
            res_trials["root_music"].append(_resolution_success(theta_root, THETA_TRUE_RAD, RES_TOL_DEG, False, True))
            res_trials["esprit"].append(_resolution_success(theta_esprit, THETA_TRUE_RAD, RES_TOL_DEG, False, True))
            res_trials["dml"].append(_resolution_success(theta_dml, THETA_TRUE_RAD, RES_TOL_DEG, False, True))
            res_trials["sml"].append(_resolution_success(theta_sml, THETA_TRUE_RAD, RES_TOL_DEG, False, True))

        CRB_q, crb_rmse_deg_q, sigma_n2_theory = stochastic_crb_two_source(
            snr_db=snr_db,
            theta_true_rad=THETA_TRUE_RAD,
            num_rx=NUM_RX,
            num_snapshots=NUM_SNAPSHOTS,
            d_over_lambda=D_OVER_LAMBDA,
            sigma_s2=1.0,
        )
        crb_curve_deg.append(float(crb_rmse_deg_q))

        for m in method_names:
            err = np.array(err_trials[m])
            rmse_deg_m = float(np.sqrt(np.mean(np.sum(err**2, axis=1) / NUM_SOURCES)) * (180.0 / np.pi))
            pres_m = float(np.mean(res_trials[m]))
            rt_ms_m = float((runtime_acc[m] / NUM_TRIALS) * 1000.0)
            ratio_m = float(rmse_deg_m / (crb_rmse_deg_q + 1e-12))
            mask_m = int(rmse_deg_m > 3.0 * crb_rmse_deg_q)

            # Physical range guards + clipping
            if not (0.0 <= rmse_deg_m <= 180.0):
                print(f"WARNING: {m} RMSE out of range ({rmse_deg_m:.3f}), clipping.")
                rmse_deg_m = float(np.clip(rmse_deg_m, 0.0, 180.0))
            if not (0.0 <= pres_m <= 1.0):
                print(f"WARNING: {m} P_res out of range ({pres_m:.3f}), clipping.")
                pres_m = float(np.clip(pres_m, 0.0, 1.0))

            rmse_curve[m].append(rmse_deg_m)
            pres_curve[m].append(pres_m)
            runtime_curve_ms[m].append(rt_ms_m)
            ratio_curve[m].append(ratio_m)
            threshold_mask[m].append(mask_m)

    threshold_snr_db = {}
    for m in method_names:
        idx = np.where(np.array(pres_curve[m]) >= RES_TARGET)[0]
        threshold_snr_db[m] = float(SNR_POINTS_DB[idx[0]]) if len(idx) > 0 else None

    snapshot_sensitivity = {"x": [float(v) for v in SNR_POINTS_DB], "x_label": "SNR (dB)"}
    for Ls in SNAPSHOT_POINTS:
        curve_ls = []
        for snr_db in SNR_POINTS_DB:
            errs = []
            for _ in range(NUM_TRIALS_SENS):
                _, Rhat_s, _, _ = generate_snapshots(
                    snr_db=snr_db,
                    num_snapshots=Ls,
                    theta_true_rad=THETA_TRUE_RAD,
                    num_rx=NUM_RX,
                    num_sources=NUM_SOURCES,
                    d_over_lambda=D_OVER_LAMBDA,
                    sigma_s2=1.0,
                )
                theta_m, _, _, _, _, _ = music_spectrum_and_estimate(
                    Rhat_s, NUM_SOURCES, theta_grid_rad, NUM_RX, D_OVER_LAMBDA
                )
                errs.append(_pair_error_rad(theta_m, THETA_TRUE_RAD))
            errs = np.array(errs)
            rmse_ls = float(np.sqrt(np.mean(np.sum(errs**2, axis=1) / NUM_SOURCES)) * (180.0 / np.pi))
            if not (0.0 <= rmse_ls <= 180.0):
                print(f"WARNING: snapshot sensitivity RMSE out of range (L={Ls}, {rmse_ls:.3f}), clipping.")
                rmse_ls = float(np.clip(rmse_ls, 0.0, 180.0))
            curve_ls.append(rmse_ls)
        snapshot_sensitivity[f"music_L{Ls}"] = curve_ls

    # Constant/flat-line diagnostics
    for method_name in method_names + ["stochastic_crb"]:
        arr = crb_curve_deg if method_name == "stochastic_crb" else rmse_curve[method_name]
        assert len(arr) == len(SNR_POINTS_DB), f"{method_name}: result length mismatch"
        if len(set(round(float(v), 8) for v in arr)) == 1:
            print(f"WARNING: {method_name} produced constant results across all operating points — likely implementation bug.")

    for method_name in method_names:
        arrp = pres_curve[method_name]
        if len(set(round(float(v), 8) for v in arrp)) == 1:
            print(f"WARNING: {method_name} resolution curve is constant across SNR.")

    if not all(v > 0 for v in crb_curve_deg):
        print("WARNING: CRB had non-positive entries; clipping to epsilon.")
        crb_curve_deg = [float(max(v, 1e-12)) for v in crb_curve_deg]

    performance_data = {
        "metric_curve": {
            "x": [float(v) for v in SNR_POINTS_DB],
            "music": [float(v) for v in rmse_curve["music"]],
            "root_music": [float(v) for v in rmse_curve["root_music"]],
            "esprit": [float(v) for v in rmse_curve["esprit"]],
            "dml": [float(v) for v in rmse_curve["dml"]],
            "sml": [float(v) for v in rmse_curve["sml"]],
            "stochastic_crb": [float(v) for v in crb_curve_deg],
            "x_label": "SNR (dB)",
            "y_label": "RMSE (deg)",
        },
        "resolution_curve": {
            "x": [float(v) for v in SNR_POINTS_DB],
            "music": [float(v) for v in pres_curve["music"]],
            "root_music": [float(v) for v in pres_curve["root_music"]],
            "esprit": [float(v) for v in pres_curve["esprit"]],
            "dml": [float(v) for v in pres_curve["dml"]],
            "sml": [float(v) for v in pres_curve["sml"]],
            "target": float(RES_TARGET),
            "x_label": "SNR (dB)",
            "y_label": "Resolution Probability",
        },
        "rmse_to_crb_ratio": {
            "x": [float(v) for v in SNR_POINTS_DB],
            "music": [float(v) for v in ratio_curve["music"]],
            "root_music": [float(v) for v in ratio_curve["root_music"]],
            "esprit": [float(v) for v in ratio_curve["esprit"]],
            "dml": [float(v) for v in ratio_curve["dml"]],
            "sml": [float(v) for v in ratio_curve["sml"]],
            "x_label": "SNR (dB)",
        },
        "runtime_per_trial_ms": {
            "x": [float(v) for v in SNR_POINTS_DB],
            "music": [float(v) for v in runtime_curve_ms["music"]],
            "root_music": [float(v) for v in runtime_curve_ms["root_music"]],
            "esprit": [float(v) for v in runtime_curve_ms["esprit"]],
            "dml": [float(v) for v in runtime_curve_ms["dml"]],
            "sml": [float(v) for v in runtime_curve_ms["sml"]],
            "x_label": "SNR (dB)",
        },
        "threshold_region_mask": {
            "x": [float(v) for v in SNR_POINTS_DB],
            "music": [int(v) for v in threshold_mask["music"]],
            "root_music": [int(v) for v in threshold_mask["root_music"]],
            "esprit": [int(v) for v in threshold_mask["esprit"]],
            "dml": [int(v) for v in threshold_mask["dml"]],
            "sml": [int(v) for v in threshold_mask["sml"]],
            "x_label": "SNR (dB)",
        },
        "resolution_threshold_snr_db": threshold_snr_db,
        "snapshot_sensitivity": snapshot_sensitivity,
        "pseudospectrum_diagnostic": {
            "snr_db": float(DIAG_SNR_DB),
            "angle_deg": [float(v) for v in theta_grid_deg],
            "music_db": [float(v) for v in (diag_spectrum_db if diag_spectrum_db is not None else np.zeros_like(theta_grid_deg))],
            "true_angles_deg": [float(v) for v in THETA_TRUE_DEG],
        },
        "sml_convergence": {
            "objective_values": [float(v) for v in diag_obj_hist_sml],
            "description": "Single diagnostic trial objective history at 20 dB",
        },
    }

    return performance_data

# AUTO-WISP RESULTS HELPER
def _autowisp_normalize_results(candidate):
    if candidate is None:
        candidate = {}
    if not isinstance(candidate, dict):
        candidate = {'performance_data': {'raw_results': candidate}}
    performance_data = candidate.get('performance_data')
    if not isinstance(performance_data, dict):
        for key in ('metrics', 'results', 'evaluation_results', 'raw_results'):
            value = candidate.get(key)
            if isinstance(value, dict):
                performance_data = value
                break
    if not isinstance(performance_data, dict):
        performance_data = {'raw_results': performance_data if performance_data is not None else {}}
    report_assets = candidate.get('report_assets')
    if not isinstance(report_assets, dict):
        report_assets = {}
    report_assets.setdefault('problem_summary', 'Recovered from notebook auto-debug path.')
    report_assets.setdefault('solution_summary', 'Execution contract was normalized to ensure RESULTS generation.')
    report_assets.setdefault('evaluation_summary', 'Primary metric exported as RMSE_deg.')
    performance_data = _autowisp_to_builtin(performance_data)
    tables = report_assets.get('tables')
    if not isinstance(tables, list) or not tables:
        tables = _autowisp_build_tables(performance_data)
    report_assets['tables'] = tables
    figures = report_assets.get('figures') or report_assets.get('figures_metadata') or []
    if not isinstance(figures, list):
        figures = []
    if not figures:
        figures = _autowisp_build_figures(performance_data)
    report_assets['figures'] = figures
    report_assets['figures_metadata'] = figures
    candidate['algorithm'] = candidate.get('algorithm') or 'Notebook Auto-Debug'
    candidate['elapsed_sec'] = float(candidate.get('elapsed_sec') or 0.0)
    candidate['performance_data'] = performance_data
    candidate['report_assets'] = report_assets
    return candidate

def _autowisp_to_builtin(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): _autowisp_to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_autowisp_to_builtin(v) for v in value]
    if hasattr(value, 'tolist'):
        return _autowisp_to_builtin(value.tolist())
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return str(value)

def _autowisp_curve_specs(perf_data):
    if not isinstance(perf_data, dict):
        return []
    _x_keys = ('x', 'variable_points', 'operating_points', 'snr', 'snr_points', 'snr_db')
    top_x = None
    for key in _x_keys:
        value = perf_data.get(key)
        if isinstance(value, list) and len(value) >= 2:
            top_x = value
            break
    specs = []
    for metric_name, payload in perf_data.items():
        if not isinstance(payload, dict):
            continue
        x_values = None
        for key in _x_keys:
            value = payload.get(key)
            if isinstance(value, list) and len(value) >= 2:
                x_values = value
                break
        if x_values is None:
            x_values = top_x
        if not isinstance(x_values, list) or len(x_values) < 2:
            continue
        _skip_keys = set(_x_keys) | {'x_label', 'xlabel', 'ylabel', 'title'}
        series = {}
        for series_name, series_values in payload.items():
            if series_name in _skip_keys:
                continue
            if isinstance(series_values, list) and len(series_values) == len(x_values):
                series[str(series_name)] = series_values
        if series:
            specs.append({
                'metric': str(metric_name),
                'x': x_values,
                'x_label': payload.get('x_label') or payload.get('xlabel') or 'Operating Point',
                'series': series,
            })
    return specs

def _autowisp_build_tables(perf_data):
    tables = []
    for spec in _autowisp_curve_specs(perf_data):
        columns = [spec['x_label']] + [name.replace('_', ' ').title() for name in spec['series'].keys()]
        rows = []
        for idx, x_value in enumerate(spec['x']):
            rows.append([x_value] + [values[idx] for values in spec['series'].values()])
        tables.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'columns': columns,
            'rows': rows,
        })
    return tables

def _autowisp_build_figures(perf_data):
    figures = []
    for spec in _autowisp_curve_specs(perf_data):
        figures.append({
            'title': spec['metric'].replace('_', ' ').title(),
            'xlabel': spec['x_label'],
            'ylabel': spec['metric'].replace('_', ' '),
            'x': spec['x'],
            'series': spec['series'],
        })
    return figures

if 'run_experiment' not in globals():
    def run_experiment(eval_config=None):
        eval_config = dict(eval_config or globals().get('EVAL_CONFIG') or {})
        existing_runner = globals().get('run_evaluation')
        if callable(existing_runner):
            return _autowisp_normalize_results(existing_runner(eval_config))
        existing_results = globals().get('RESULTS')
        if isinstance(existing_results, dict):
            return _autowisp_normalize_results(existing_results)
        for key in ('performance_data', 'metrics', 'results', 'evaluation_results'):
            value = globals().get(key)
            if isinstance(value, dict):
                return _autowisp_normalize_results({'performance_data': value})
        raise RuntimeError('Notebook auto-debug could not infer performance_data for RESULTS generation')

Next block runs the experiment with fixed reproducible hyperparameters and creates the required `RESULTS` dictionary.

**Computes:**
- sets seed (`np.random.seed(42)`),
- defines all simulation hyperparameters by name,
- calls `run_evaluation(eval_config)`,
- builds `RESULTS = {algorithm, elapsed_sec, performance_data, report_assets}`.

**Key inputs/outputs:**
- Input: `EVAL_CONFIG`
- Output: `RESULTS` with JSON-serializable performance curves and report assets.

**Equations:**
$
\gamma=10^{\mathrm{SNR}_{\mathrm{dB}}/10},\ 
\mathrm{SNR}_{\mathrm{th}}=\min\{\mathrm{SNR}:P_{\mathrm{res}}\ge0.9\}
$.

In [ ]:
np.random.seed(42)

# Named hyperparameters (D10)
NUM_RX = 8
NUM_SOURCES = 2
NUM_SNAPSHOTS = 200
D_OVER_LAMBDA = 0.5
THETA_TRUE_DEG = [-7.5, 7.5]
SNR_POINTS_DB = list(np.arange(0, 31, 2))
NUM_MONTE_CARLO = 1000
GRID_STEP_DEG = 0.1
SCAN_RANGE_DEG = [-90, 90]
RES_TOL_DEG = 2.0
RES_TARGET = 0.9
SNAPSHOT_POINTS = [50, 100, 200, 400]
DIAG_SNR_DB = 20.0

EVAL_CONFIG = {
    "snr_points": SNR_POINTS_DB,
    "num_trials": NUM_MONTE_CARLO,
    "num_rx": NUM_RX,
    "num_sources": NUM_SOURCES,
    "num_snapshots": NUM_SNAPSHOTS,
    "d_over_lambda": D_OVER_LAMBDA,
    "theta_true_deg": THETA_TRUE_DEG,
    "scan_range_deg": SCAN_RANGE_DEG,
    "grid_step_deg": GRID_STEP_DEG,
    "resolution_tol_deg": RES_TOL_DEG,
    "resolution_prob_target": RES_TARGET,
    "snapshot_points": SNAPSHOT_POINTS,
    "num_trials_sensitivity": 250,
    "diagnostic_snr_db": DIAG_SNR_DB,
}

t_exec0 = time.perf_counter()
performance_data = run_evaluation(EVAL_CONFIG)
elapsed_sec = float(time.perf_counter() - t_exec0)

key_snr_points = [0, 10, 20, 30]
snr_x = np.array(performance_data["metric_curve"]["x"], dtype=float)
key_idx = {k: int(np.argmin(np.abs(snr_x - k))) for k in key_snr_points}

methods_for_table = ["music", "root_music", "esprit", "dml", "sml"]
table_rows = []
for m in methods_for_table:
    row = {"method": m.upper()}
    for s in key_snr_points:
        row[f"rmse@{s}dB"] = float(performance_data["metric_curve"][m][key_idx[s]])
        row[f"pres@{s}dB"] = float(performance_data["resolution_curve"][m][key_idx[s]])
    row["threshold_snr_db"] = performance_data["resolution_threshold_snr_db"][m]
    table_rows.append(row)

RESULTS = {
    "algorithm": "Grid-MUSIC with Root-MUSIC/ESPRIT/DML/SML baselines and stochastic CRB",
    "elapsed_sec": elapsed_sec,
    "performance_data": performance_data,
    "report_assets": {
        "problem_summary": "Two-source DoA estimation for 8-element half-wavelength ULA, L=200 snapshots, SNR sweep 0:2:30 dB.",
        "solution_summary": "MUSIC pseudo-spectrum localization benchmarked against Root-MUSIC, ESPRIT, DML, SML, and stochastic CRB with Monte Carlo averaging.",
        "evaluation_summary": "Includes RMSE-vs-SNR, resolution probability-vs-SNR, CRB ratio, threshold masks, runtime, pseudo-spectrum diagnostics, and snapshot sensitivity.",
        "tables": [
            {
                "name": "rmse_resolution_key_points",
                "rows": table_rows
            }
        ],
        "figures": []
    }
}

# AUTO-WISP EXECUTION CONTRACT
if not isinstance(globals().get('EVAL_CONFIG'), dict):
    EVAL_CONFIG = {}
if not isinstance(globals().get('RESULTS'), dict):
    RESULTS = run_experiment(globals().get('EVAL_CONFIG', {}))
RESULTS = _autowisp_normalize_results(RESULTS)
RESULTS.setdefault('notes', {})['primary_metric'] = 'RMSE_deg'

Next block creates and saves all required plots:
1. RMSE vs SNR (semilogy) for all methods + CRB,
2. Resolution probability vs SNR (linear) with 0.9 target,
3. MUSIC pseudo-spectrum at 20 dB,
4. Snapshot sensitivity RMSE vs SNR for $L=\{50,100,200,400\}$.

**Inputs/outputs:**
- Input: `RESULTS["performance_data"]`
- Output: saved PNG figures and figure entries appended to `RESULTS["report_assets"]["figures"]`.

**Equations:**
$
P_{\mathrm{MU,dB}}(\theta)=10\log_{10}\left(P_{\mathrm{MU}}(\theta)/\max_\theta P_{\mathrm{MU}}(\theta)\right)
$.

In [ ]:
perf = RESULTS["performance_data"]

snr = np.array(perf["metric_curve"]["x"], dtype=float)

# Fig 1: RMSE vs SNR (semilogy)
fig1, ax1 = plt.subplots(figsize=(8, 5))
ax1.semilogy(snr, perf["metric_curve"]["music"], "o-", label="MUSIC")
ax1.semilogy(snr, perf["metric_curve"]["root_music"], "s-", label="Root-MUSIC")
ax1.semilogy(snr, perf["metric_curve"]["esprit"], "d-", label="ESPRIT")
ax1.semilogy(snr, perf["metric_curve"]["dml"], "^-", label="DML")
ax1.semilogy(snr, perf["metric_curve"]["sml"], "v-", label="SML")
ax1.semilogy(snr, perf["metric_curve"]["stochastic_crb"], "k--", linewidth=2, label="Stochastic CRB")
ax1.set_xlabel("SNR (dB)")
ax1.set_ylabel("RMSE (deg)")
ax1.set_title("DoA RMSE vs SNR")
ax1.grid(True, which="both", ls=":")
ax1.legend()
fig1.tight_layout()
fig1.savefig("fig1_rmse_vs_snr.png", dpi=150, bbox_inches="tight")
RESULTS["report_assets"]["figures"].append("fig1_rmse_vs_snr.png")

# Fig 2: Resolution probability vs SNR
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.plot(snr, perf["resolution_curve"]["music"], "o-", label="MUSIC")
ax2.plot(snr, perf["resolution_curve"]["root_music"], "s-", label="Root-MUSIC")
ax2.plot(snr, perf["resolution_curve"]["esprit"], "d-", label="ESPRIT")
ax2.plot(snr, perf["resolution_curve"]["dml"], "^-", label="DML")
ax2.plot(snr, perf["resolution_curve"]["sml"], "v-", label="SML")
ax2.axhline(perf["resolution_curve"]["target"], color="k", linestyle="--", label="Target 0.9")
ax2.set_xlabel("SNR (dB)")
ax2.set_ylabel("Resolution Probability")
ax2.set_ylim([0, 1.02])
ax2.set_title("Resolution Probability vs SNR")
ax2.grid(True, ls=":")
ax2.legend()
fig2.tight_layout()
fig2.savefig("fig2_resolution_probability.png", dpi=150, bbox_inches="tight")
RESULTS["report_assets"]["figures"].append("fig2_resolution_probability.png")

# Fig 3: MUSIC pseudo-spectrum at 20 dB
angles = np.array(perf["pseudospectrum_diagnostic"]["angle_deg"], dtype=float)
music_db = np.array(perf["pseudospectrum_diagnostic"]["music_db"], dtype=float)
true_angles = perf["pseudospectrum_diagnostic"]["true_angles_deg"]

fig3, ax3 = plt.subplots(figsize=(8, 5))
ax3.plot(angles, music_db, "b-", linewidth=1.5, label="Single-trial MUSIC spectrum")
for i, ta in enumerate(true_angles):
    ax3.axvline(ta, color="r", linestyle="--", label="True DoA" if i == 0 else None)
ax3.set_xlabel("Angle (deg)")
ax3.set_ylabel("Pseudo-spectrum (dB, normalized)")
ax3.set_title(f"MUSIC Pseudo-spectrum @ {perf['pseudospectrum_diagnostic']['snr_db']} dB")
ax3.grid(True, ls=":")
ax3.legend()
fig3.tight_layout()
fig3.savefig("fig3_music_pseudospectrum_20db.png", dpi=150, bbox_inches="tight")
RESULTS["report_assets"]["figures"].append("fig3_music_pseudospectrum_20db.png")

# Fig 4: Sensitivity RMSE vs SNR for multiple snapshots (semilogy)
fig4, ax4 = plt.subplots(figsize=(8, 5))
ax4.semilogy(snr, perf["snapshot_sensitivity"]["music_L50"], "o-", label="MUSIC L=50")
ax4.semilogy(snr, perf["snapshot_sensitivity"]["music_L100"], "s-", label="MUSIC L=100")
ax4.semilogy(snr, perf["snapshot_sensitivity"]["music_L200"], "d-", label="MUSIC L=200")
ax4.semilogy(snr, perf["snapshot_sensitivity"]["music_L400"], "^-", label="MUSIC L=400")
ax4.set_xlabel("SNR (dB)")
ax4.set_ylabel("RMSE (deg)")
ax4.set_title("Snapshot Sensitivity: MUSIC RMSE vs SNR")
ax4.grid(True, which="both", ls=":")
ax4.legend()
fig4.tight_layout()
fig4.savefig("fig4_snapshot_sensitivity.png", dpi=150, bbox_inches="tight")
RESULTS["report_assets"]["figures"].append("fig4_snapshot_sensitivity.png")

### Result Notes

- The primary comparison is in `RESULTS["performance_data"]["metric_curve"]`.
- Resolution-threshold SNRs are in `RESULTS["performance_data"]["resolution_threshold_snr_db"]`.
- Threshold-region flags (`RMSE > 3x CRB`) are in `RESULTS["performance_data"]["threshold_region_mask"]`.
- Snapshot sensitivity is stored in `RESULTS["performance_data"]["snapshot_sensitivity"]`.

### Method Comparison Table (key operating points)

| Method | RMSE@0dB | RMSE@10dB | RMSE@20dB | RMSE@30dB | P_res@0dB | P_res@10dB | P_res@20dB | P_res@30dB | Resolution Threshold SNR (dB) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| MUSIC | `metric_curve.music[idx0]` | `metric_curve.music[idx10]` | `metric_curve.music[idx20]` | `metric_curve.music[idx30]` | `resolution_curve.music[idx0]` | `...` | `...` | `...` | `resolution_threshold_snr_db.music` |
| Root-MUSIC | `metric_curve.root_music[idx0]` | `...` | `...` | `...` | `resolution_curve.root_music[idx0]` | `...` | `...` | `...` | `resolution_threshold_snr_db.root_music` |
| ESPRIT | `metric_curve.esprit[idx0]` | `...` | `...` | `...` | `resolution_curve.esprit[idx0]` | `...` | `...` | `...` | `resolution_threshold_snr_db.esprit` |
| DML | `metric_curve.dml[idx0]` | `...` | `...` | `...` | `resolution_curve.dml[idx0]` | `...` | `...` | `...` | `resolution_threshold_snr_db.dml` |
| SML | `metric_curve.sml[idx0]` | `...` | `...` | `...` | `resolution_curve.sml[idx0]` | `...` | `...` | `...` | `resolution_threshold_snr_db.sml` |

Use `RESULTS["report_assets"]["tables"][0]["rows"]` for programmatically filled values at $\{0,10,20,30\}$ dB.

# estimation|localization

This notebook implements the **Subspace pseudo-spectrum estimation with multi-baseline Monte Carlo SNR sweep** solution as a single executable workflow.

## Problem Setup

We consider the **estimation|localization** task under a notebook-first experimental workflow. The working waveform assumption is **custom_narrowband**, and the reference channel or propagation setting is **free_space_los**.

### Objective

Estimate or recover the task-dependent latent quantity while tracking the primary metric **RMSE_deg** across the configured evaluation protocol.

### Evaluation Configuration

- Evaluation axis: SNR (dB)
- Operating points: ['0', '2', '4', '6', '8', '10', '12', '14', '16', '18', '20', '22', '24', '26', '28', '30']
- Monte Carlo trials: 1000
- Notebook artifact flow: problem summary -> executable experiment -> report assets

## AutoWiSP Simulation Outputs

- Algorithm: Grid-MUSIC with Root-MUSIC/ESPRIT/DML/SML baselines and stochastic CRB
- Verification status: passed
- Simulation status: success
- Execution time: 243.28363274998264

### Evaluation Summary

Includes RMSE-vs-SNR, resolution probability-vs-SNR, CRB ratio, threshold masks, runtime, pseudo-spectrum diagnostics, and snapshot sensitivity.

### Validation Notes

- **Verification status**: passed
- **Simulation status**: success
- **Performance payload keys**: metric_curve, resolution_curve, rmse_to_crb_ratio, runtime_per_trial_ms, threshold_region_mask, resolution_threshold_snr_db, snapshot_sensitivity, pseudospectrum_diagnostic, sml_convergence


## Quantitative Tables

### Metric Curve
| SNR (dB) | Music | Root Music | Esprit | Dml | Sml | Stochastic Crb |
| --- | --- | --- | --- | --- | --- | --- |
| 0.0 | 0.2615817271906656 | 0.26021754215945253 | 0.32615926719737837 | 0.261495697861457 | 0.261495697861457 | 0.17966659186013026 |
| 2.0 | 0.20386269889299286 | 0.20204604148015973 | 0.24663554530666318 | 0.20386269889299286 | 0.20386269889299286 | 0.13874886159693756 |
| 4.0 | 0.1548063306200692 | 0.15247560095339416 | 0.18950391768118385 | 0.1548063306200692 | 0.1548063306200692 | 0.10823038122794973 |
| 6.0 | 0.12515989773085642 | 0.12180623954652957 | 0.15093866072430895 | 0.12515989773085642 | 0.12515989773085642 | 0.08498402946864075 |
| 8.0 | 0.10259142264337338 | 0.09768295635240401 | 0.11944241262405063 | 0.10259142264337338 | 0.10259142264337338 | 0.06701439478278079 |
| 10.0 | 0.08105553651661541 | 0.07544930473359765 | 0.09244567559060832 | 0.08105553651661541 | 0.08105553651661541 | 0.05298691019913648 |
| 12.0 | 0.06621933252463494 | 0.059982259897232396 | 0.07361762636064581 | 0.06621933252463494 | 0.06621933252463494 | 0.04196699854838438 |
| 14.0 | 0.0522015325445789 | 0.0470412326679191 | 0.05881921442430415 | 0.0522015325445789 | 0.0522015325445789 | 0.03327461785951832 |
| 16.0 | 0.043646305685623686 | 0.03776657722705851 | 0.046438374788697556 | 0.043646305685623686 | 0.043646305685623686 | 0.026400483239689528 |
| 18.0 | 0.030166206257845263 | 0.029947955013398857 | 0.03695377346645725 | 0.030166206257845263 | 0.030166206257845263 | 0.020955391505733174 |
| 20.0 | 0.01884144368143271 | 0.023533217571879247 | 0.02875693122577785 | 0.01884144368143271 | 0.01884144368143271 | 0.016637818977481518 |
| 22.0 | 0.008366600265207838 | 0.018771374238040556 | 0.02279514413223492 | 0.008366600265207838 | 0.008366600265207838 | 0.013212062413778734 |
| 24.0 | 0.004472135955123245 | 0.014791710453789004 | 0.01821505347258126 | 0.004472135955123245 | 0.004472135955123245 | 0.010492796887294036 |
| 26.0 | 5.133644087158774e-12 | 0.012121830922917885 | 0.014771972265196836 | 5.133644087158774e-12 | 5.133644087158774e-12 | 0.00833376411129539 |
| 28.0 | 5.133644087158774e-12 | 0.009039880116077337 | 0.011127456576085836 | 5.133644087158774e-12 | 5.133644087158774e-12 | 0.006619262706329667 |
| 30.0 | 5.133644087158774e-12 | 0.007445947377927411 | 0.00909699715033535 | 5.133644087158774e-12 | 5.133644087158774e-12 | 0.005257625997084685 |

### Resolution Curve
| SNR (dB) | Music | Root Music | Esprit | Dml | Sml |
| --- | --- | --- | --- | --- | --- |
| 0.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 2.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 4.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 6.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 8.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 10.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 12.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 14.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 16.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 18.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 20.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 22.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 24.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 26.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 28.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |
| 30.0 | 1.0 | 1.0 | 1.0 | 1.0 | 1.0 |

### Rmse To Crb Ratio
| SNR (dB) | Music | Root Music | Esprit | Dml | Sml |
| --- | --- | --- | --- | --- | --- |
| 0.0 | 1.4559285868396168 | 1.448335717085247 | 1.815358458235111 | 1.4554497592049551 | 1.4554497592049551 |
| 2.0 | 1.4692927678480008 | 1.4561996340239742 | 1.7775680640995566 | 1.4692927678480008 | 1.4692927678480008 |
| 4.0 | 1.430340805070187 | 1.408805912185123 | 1.750930889546704 | 1.430340805070187 | 1.430340805070187 |
| 6.0 | 1.4727460972601667 | 1.4332838805912702 | 1.776082655367965 | 1.4727460972601667 | 1.4727460972601667 |
| 8.0 | 1.5308863561982529 | 1.4576414017850057 | 1.7823396452273679 | 1.5308863561982529 | 1.5308863561982529 |
| 10.0 | 1.529727553662992 | 1.4239234642786043 | 1.7446889286699756 | 1.529727553662992 | 1.529727553662992 |
| 12.0 | 1.5778906000797697 | 1.429272094039527 | 1.7541789716988403 | 1.5778906000797697 | 1.5778906000797697 |
| 14.0 | 1.5688093778687129 | 1.4137272098843674 | 1.767690155627468 | 1.5688093778687129 | 1.5688093778687129 |
| 16.0 | 1.6532388929288302 | 1.4305259825263763 | 1.7589971503674902 | 1.6532388929288302 | 1.6532388929288302 |
| 18.0 | 1.4395439115586346 | 1.429128871382636 | 1.7634494423348588 | 1.4395439115586346 | 1.4395439115586346 |
| 20.0 | 1.1324467290935938 | 1.4144412559311939 | 1.7284075071961391 | 1.1324467290935938 | 1.1324467290935938 |
| 22.0 | 0.6332546730818601 | 1.420775473861166 | 1.7253282202736762 | 0.6332546730818601 | 0.6332546730818601 |
| 24.0 | 0.4262100946709876 | 1.4097013990893998 | 1.7359578829646767 | 0.4262100946709876 | 0.4262100946709876 |
| 26.0 | 6.160054469966035e-10 | 1.4545445202887002 | 1.7725450428099714 | 6.160054469966035e-10 | 6.160054469966035e-10 |
| 28.0 | 7.755613146271061e-10 | 1.3656929050522895 | 1.681071906054452 | 7.755613146271061e-10 | 7.755613146271061e-10 |
| 30.0 | 9.764186515033445e-10 | 1.4162185329728503 | 1.730248053712706 | 9.764186515033445e-10 | 9.764186515033445e-10 |

### Runtime Per Trial Ms
| SNR (dB) | Music | Root Music | Esprit | Dml | Sml |
| --- | --- | --- | --- | --- | --- |
| 0.0 | 4.652651005832013 | 0.12717889924533665 | 0.050000734452623874 | 3.6102354573085904 | 1.9073730598320253 |
| 2.0 | 4.668037740048021 | 0.12297536357073113 | 0.04786872223485261 | 3.594778154452797 | 1.9042906292015687 |
| 4.0 | 4.627360337937716 | 0.1233645171741955 | 0.04805642564315349 | 3.593838417902589 | 1.8982905521988869 |
| 6.0 | 4.630839500867296 | 0.12329669296741484 | 0.048012080311309546 | 3.614855050109327 | 1.909397047071252 |
| 8.0 | 4.635881415801123 | 0.1233199283014983 | 0.04800506436731666 | 3.6447185974102467 | 1.923414025863167 |
| 10.0 | 4.630231614341028 | 0.12296708644134922 | 0.04758053971454501 | 3.6141933155595325 | 1.9025670675910078 |
| 12.0 | 4.665855477389414 | 0.12210838665487246 | 0.04716749343788251 | 3.583368216524832 | 1.881049090938177 |
| 14.0 | 4.643183831882197 | 0.12367519061081111 | 0.04791927826590836 | 3.609522469807416 | 1.8951458916417323 |
| 16.0 | 4.637519353302196 | 0.1237629140377976 | 0.04806827590800822 | 3.62293655925896 | 1.904879427747801 |
| 18.0 | 4.6375869196490385 | 0.12277885596267879 | 0.047644537524320185 | 3.5969177669030614 | 1.8872508746571839 |
| 20.0 | 4.692802599456627 | 0.1329290111316368 | 0.0527791166678071 | 3.6086784919025376 | 1.9183658582041971 |
| 22.0 | 4.653401053394191 | 0.13530569145223126 | 0.053626270790118724 | 3.568571057054214 | 1.9114929886418395 |
| 24.0 | 4.651762583351228 | 0.12819495826261118 | 0.05010645883157849 | 3.542181206168607 | 1.8763934448943473 |
| 26.0 | 4.708004870743025 | 0.13076112157432362 | 0.05115108523750678 | 3.6131746331229806 | 1.9072309985058382 |
| 28.0 | 4.617896900279447 | 0.1259687290294096 | 0.048705462482757866 | 3.534794211096596 | 1.8663032434415072 |
| 30.0 | 4.627818459470291 | 0.12743551941821352 | 0.04941808251896873 | 3.528551845345646 | 1.8655907584470697 |

### Threshold Region Mask
| SNR (dB) | Music | Root Music | Esprit | Dml | Sml |
| --- | --- | --- | --- | --- | --- |
| 0.0 | 0 | 0 | 0 | 0 | 0 |
| 2.0 | 0 | 0 | 0 | 0 | 0 |
| 4.0 | 0 | 0 | 0 | 0 | 0 |
| 6.0 | 0 | 0 | 0 | 0 | 0 |
| 8.0 | 0 | 0 | 0 | 0 | 0 |
| 10.0 | 0 | 0 | 0 | 0 | 0 |
| 12.0 | 0 | 0 | 0 | 0 | 0 |
| 14.0 | 0 | 0 | 0 | 0 | 0 |
| 16.0 | 0 | 0 | 0 | 0 | 0 |
| 18.0 | 0 | 0 | 0 | 0 | 0 |
| 20.0 | 0 | 0 | 0 | 0 | 0 |
| 22.0 | 0 | 0 | 0 | 0 | 0 |
| 24.0 | 0 | 0 | 0 | 0 | 0 |
| 26.0 | 0 | 0 | 0 | 0 | 0 |
| 28.0 | 0 | 0 | 0 | 0 | 0 |
| 30.0 | 0 | 0 | 0 | 0 | 0 |

### Snapshot Sensitivity
| SNR (dB) | Music L50 | Music L100 | Music L200 | Music L400 |
| --- | --- | --- | --- | --- |
| 0.0 | 0.5502544865784895 | 0.35527454172809025 | 0.26754438884024767 | 0.19230184606485265 |
| 2.0 | 0.4332666615373235 | 0.2795711000802073 | 0.1975348070593422 | 0.141703916670938 |
| 4.0 | 0.3252998616665711 | 0.21949943052310583 | 0.16751119365625167 | 0.11081516141747731 |
| 6.0 | 0.26026909151870165 | 0.18368451213983225 | 0.12449899597987824 | 0.09066421565309386 |
| 8.0 | 0.19712939912637806 | 0.1426884718541255 | 0.10059821071965172 | 0.0726636084986268 |
| 10.0 | 0.16248076809259246 | 0.1115347479484055 | 0.0809938269251692 | 0.06260990336963117 |
| 12.0 | 0.12049896265149487 | 0.08988882021636486 | 0.061644140030162206 | 0.049598387070262574 |
| 14.0 | 0.09889388252063876 | 0.0728010988925752 | 0.05329165037741718 | 0.033166247903246196 |
| 16.0 | 0.07861297602806144 | 0.058309518948484934 | 0.04171330722874991 | 0.02683281572993244 |
| 18.0 | 0.06542170893486812 | 0.047958315233140696 | 0.030331501776227 | 0.012649110640848062 |
| 20.0 | 0.05138093031468099 | 0.03464101615127225 | 0.01843908891442292 | 0.00774596669255749 |
| 22.0 | 0.04289522117867839 | 0.0223606797752445 | 0.0044721359552471526 | 5.133644087158774e-12 |
| 24.0 | 0.03162277660198949 | 0.011832159565946356 | 0.0044721359552471526 | 5.133644087158774e-12 |
| 26.0 | 0.019999999999862565 | 0.007745966692513555 | 5.133644087158774e-12 | 5.133644087158774e-12 |
| 28.0 | 0.009999999999939679 | 5.133644087158774e-12 | 5.133644087158774e-12 | 5.133644087158774e-12 |
| 30.0 | 5.133644087158774e-12 | 5.133644087158774e-12 | 5.133644087158774e-12 | 5.133644087158774e-12 |


## Figures and Comparison Plots

![Asset Curve 1](figures/asset_curve_1.png)

![Asset Curve 2](figures/asset_curve_2.png)

![Asset Curve 3](figures/asset_curve_3.png)

![Asset Curve 4](figures/asset_curve_4.png)

![Asset Curve 5](figures/asset_curve_5.png)

![Asset Curve 6](figures/asset_curve_6.png)

![Perf Curve 10](figures/perf_curve_10.png)

![Perf Curve 11](figures/perf_curve_11.png)

![Perf Curve 12](figures/perf_curve_12.png)

![Perf Curve 7](figures/perf_curve_7.png)

![Perf Curve 8](figures/perf_curve_8.png)

![Perf Curve 9](figures/perf_curve_9.png)
